# Data


In order to run the bellow cells, download Amazon datasets for electronics from https://amazon-reviews-2023.github.io/main.html and place them in the `/data` folder

In [ ]:
import json
import gzip
import pandas as pd

In [ ]:
with gzip.open("../../data/meta_Books.jsonl.gz") as f:
    first_line = json.loads(f.readline())

In [ ]:
first_line

## Filter items that were published after 2022

In [ ]:
def filter_out_data(data: dict) -> bool:
    filter = False
    if int(data['details']['Publisher'][-5:-1]) < 2022:
        filter = True
    
    return filter

In [ ]:
with gzip.open("../../data/meta_Books.jsonl.gz", 'rt') as fp:
    with open("../../data/meta_Books_2022_2023.jsonl", 'a', encoding='utf-8') as fp_out:
        with open("../../data/meta_Books_2022_2023_no_data.jsonl", 'a', encoding='utf-8') as fp_out_no_data:
            i = 0
            for line in fp:
                data = json.loads(line.strip())
                try:
                    if not filter_out_data(data):
                        json.dump(data, fp_out)
                        fp_out.write("\n")
                        fp_out.flush()
                
                except:
                    json.dump(data, fp_out_no_data)
                    fp_out_no_data.write("\n")
                    fp_out_no_data.flush()
                
                i += 1
                if i % 10_000 == 0:
                    print(f"Processed {i} lines")

### Filter books with Main Category == `Books`

Exclude Kindle books

In [ ]:
def filter_category(data: dict) -> bool:
    filter = False
    if len(data['categories']) < 2
        filter = True

    return filter

In [ ]:
with open("../../data/meta_Books_2022_2023.jsonl", 'rt') as fp:
    with open("../../data/meta_Books_2022_2023_Books.jsonl", 'a', encoding='utf-8') as fp_out:
        with open("../../data/meta_Books_2022_2023_not_Books.jsonl", 'a', encoding='utf-8') as fp_out_no_data:
            i = 0
            for line in fp:
                data = json.loads(line.strip())
                try:
                    if not filter_category(data):
                        json.dump(data, fp_out)
                        fp_out.write("\n")
                        fp_out.flush()
                
                except:
                    json.dump(data, fp_out_no_data)
                    fp_out_no_data.write("\n")
                    fp_out_no_data.flush()
                
                i += 1
                if i % 10_000 == 0:
                    print(f"Processed {i} lines")

### Explore data distributions

In [ ]:
df = pd.read_json("../../data/meta_Books_2022_2023_Books.jsonl", lines=True)

In [ ]:
df

In [ ]:
df['categories_middle'] = df['categories'].apply(lambda r: r[1] if len(r) > 1 else None)

In [ ]:
df['categories_middle'].value_counts().plot(kind="bar")

In [ ]:
df[df['rating_number'] < 100]['rating_number'].value_counts()

### Filter Books that have at least 10 ratings

In [ ]:
df_ratings_10 = df[df['rating_number'] >= 10]

In [ ]:
df_ratings_10['categories_middle'].value_counts().plot(kind="bar")

In [ ]:
df_ratings_10['average_rating'].plot(kind='hist', bins=50, range=(0, 5))

### Sample 1000

In [ ]:
df_sample_1000 = df_ratings_10.sample(n=1000, random_state=42)

In [ ]:
len(df_sample_1000)

In [ ]:
df_sample_1000['average_rating'].plot(kind='hist', bins=50, range=(0, 5))

In [ ]:
df_ratings_10.to_json("../../data/meta_Books_2022_2023_with_category_ratings_10.jsonl", orient='records', lines=True)
df_sample_1000.to_json("../../data/meta_Books_2022_2023_with_category_ratings_10_sample_1000.jsonl", orient='records', lines=True)

### Extract reviews that match sample data

In [ ]:
df_ratings_10 = pd.read_json("../../data/meta_Books_2022_2023_with_category_ratings_10.jsonl", lines=True)
df_sample_1000 = pd.read_json("../../data/meta_Books_2022_2023_with_category_ratings_10_sample_1000.jsonl", lines=True)

In [ ]:
with gzip.open("../../data/Books.jsonl.gz", 'rt') as fp:
    with open('../../data/Books_2022_2023_with_category_ratings_10.jsonl', 'a') as fp_out:
        id_list = set(df_ratings_10['parent_asin'].values)
        i = 0
        for line in fp:
            data = json.loads(line.strip())
            if data['parent_asin'] in id_list:
                json.dump(data, fp_out)
                fp_out.write("\n")
                fp_out.flush()
            i += 1
            if i % 100_000 == 0:
                print(f"Processed {i} lines")

In [ ]:
with open('../../data/Books_2022_2023_with_category_ratings_10.jsonl', 'rt') as fp:
    with open('../../data/Books_2022_2023_with_category_ratings_10_sample_1000.jsonl', 'a') as fp_out:
        id_list = set(df_sample_1000['parent_asin'].values)
        i = 0
        for line in fp:
            data = json.loads(line.strip())
            if data['parent_asin'] in id_list:
                json.dump(data, fp_out)
                fp_out.write("\n")
                fp_out.flush()
            i += 1
            if i % 100_000 == 0:
                print(f"Processed {i} lines")